# Priority — API walkthrough

This notebook is the demo link for the handbook submission. It uses the fictional company list in `frontend/public/sample-leads.csv` (42 rows, safe to commit) and the same FastAPI routes the React app calls.

A searcher states a buy box. The API returns which scraped companies deserve an enrichment credit.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

from app.csvparse import parse_lead_csv
from app.scoring import DEFAULT_BUY_BOX, rank_leads, summarize
from fastapi.testclient import TestClient
from app.main import app

CSV = ROOT / "frontend" / "public" / "sample-leads.csv"
print("Dataset:", CSV.name)
print("Buy box:", DEFAULT_BUY_BOX["industries"], DEFAULT_BUY_BOX["states"])

Dataset: sample-leads.csv
Buy box: ['HVAC', 'Plumbing', 'Electrical'] ['TX', 'FL', 'AZ', 'GA', 'NC']


In [2]:
leads = parse_lead_csv(CSV.read_text(encoding="utf-8"))
print(f"{len(leads)} companies")
for lead in leads[:3]:
    print(f"- {lead.company} | {lead.industry} | {lead.city}, {lead.state} | revenue={lead.revenue}")

42 companies
- Mesa Air Mechanical | HVAC | Mesa, AZ | revenue=3200000
- Hill Country Plumbing | Plumbing | Austin, TX | revenue=2100000
- Peachtree Electric | Electrical | Atlanta, GA | revenue=5400000


In [3]:
scored = rank_leads(leads, DEFAULT_BUY_BOX)
summary = summarize(scored)
print(summary)
print(f"Spend {summary['enrich']} credits. Hold back {summary['creditsHeld']}.")

{'total': 42, 'enrich': 20, 'hold': 6, 'skip': 16, 'creditsHeld': 22}
Spend 20 credits. Hold back 22.


In [4]:
by_name = {row["company"]: row for row in scored}
for name in ["Mesa Air Mechanical", "Oak City HVAC", "Mister Sparky Local"]:
    row = by_name[name]
    print(f"{row['bucket'].upper():6} {row['score']:3}  {name}")
    print(f"       {row['angle']}")
    print()

ENRICH 100  Mesa Air Mechanical
       Mesa Air Mechanical fits the buy box: HVAC in Mesa, AZ, inside the size band. Enrich the owner and open with one local fact, succession, or route density.

HOLD    78  Oak City HVAC
       Hold the credit on Oak City HVAC. Revenue is missing. Do not spend a credit until size is confirmed.

SKIP    24  Mister Sparky Local
       Skip Mister Sparky Local. Franchise. Excluded from this buy box.



## API

`POST /api/score` is what the React screen calls. `POST /api/runs` saves the ranking and returns the existing row when the same buy box and list are sent again.

In [5]:
client = TestClient(app)
parsed = client.post("/api/parse", json={"csv": CSV.read_text(encoding="utf-8")})
parsed.raise_for_status()
payload = {"buyBox": DEFAULT_BUY_BOX, "leads": parsed.json()["leads"]}

scored_response = client.post("/api/score", json=payload)
scored_response.raise_for_status()
print("POST /api/score", scored_response.status_code)
print(scored_response.json()["summary"])
print(scored_response.json()["thesis"])

saved = client.post("/api/runs", json=payload)
saved.raise_for_status()
again = client.post("/api/runs", json=payload)
again.raise_for_status()
print("POST /api/runs", saved.json()["id"][:8], "cached:", again.json()["cached"])

POST /api/score 200
{'total': 42, 'enrich': 20, 'hold': 6, 'skip': 16, 'creditsHeld': 22}
HVAC, Plumbing, Electrical in TX, FL, AZ, GA, NC, $1M–$8M revenue, 10–75 people.
POST /api/runs 2b9f1536 cached: True


In [6]:
dental = {**DEFAULT_BUY_BOX, "industries": ["Dental"]}
dental_response = client.post("/api/score", json={"buyBox": dental, "leads": payload["leads"]})
dental_response.raise_for_status()
print(dental_response.json()["thesis"])
print(dental_response.json()["summary"])

Dental in TX, FL, AZ, GA, NC, $1M–$8M revenue, 10–75 people.
{'total': 42, 'enrich': 1, 'hold': 1, 'skip': 40, 'creditsHeld': 41}
